# MH-SoyaHealthVision - clean run (leaf + UAV)
Runtime: Colab **T4 GPU**. Every step is a script in `soya_pipeline/` (see its README); this notebook only calls them.

* **Leaf**: 5-fold cross-validation, EfficientNet-B0 @ 320 px, every image is tested exactly once.
* **UAV**: frames come from a few videos recorded on different days, so only grouped 3-fold CV is reported, plus the audit that shows how far the numbers can be trusted.

## 0. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp /content/drive/MyDrive/soya_pipeline.zip /content/ && unzip -qo /content/soya_pipeline.zip -d /content
%cd /content/soya_pipeline
!pip install -q scikit-image opencv-python-headless

/content/soya_pipeline


In [3]:
import glob, os
if not glob.glob('/content/data/**/*.jpg', recursive=True):          # download + unzip only once
    !mkdir -p /content/data && cd /content/data && wget -q -O soya.zip "https://data.mendeley.com/public-api/zip/hkbgh5s3b7/download/1" && unzip -q soya.zip && rm soya.zip
    !find /content/data -name "*.zip" -execdir unzip -q -o {} \; -delete
os.environ['SOYA_ROOT'] = os.path.dirname(glob.glob('/content/data/**/Soyabean_Leaf_Image_Dataset', recursive=True)[0])
print(os.environ['SOYA_ROOT'])
!nvidia-smi -L

/content/data/MH-SoyaHealthVision An Indian UAV and Leaf Image Dataset for Integrated Crop Health Assessment
GPU 0: Tesla T4 (UUID: GPU-f46eae67-120a-a56f-e23b-4e68b4482663)


## 1. Leaf dataset
Steps 1-5: split + folds, cache once at 320 px, EDA tables, features for all images, split audit.

In [4]:
!python 01_build_manifest.py --dataset leaf --img-size 320
!python 02_cache_images.py   --dataset leaf --img-size 320
!python 03_eda.py            --dataset leaf --img-size 320
!python 04_extract_features.py --dataset leaf --img-size 320
!python 05_audit_split.py    --dataset leaf --img-size 320

Found 2782 images in /content/data/MH-SoyaHealthVision An Indian UAV and Leaf Image Dataset for Integrated Crop Health Assessment/Soyabean_Leaf_Image_Dataset
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done 740 tasks      | elapsed:   32.6s
[Parallel(n_jobs=-1)]: Done 2780 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-1)]: Done 2782 out of 2782 | elapsed:  1.8min finished

Classes (VERIFY these are your disease / pest / healthy folders):
                                        images  source_groups  clusters  dates  largest_cluster
label                                                                                          
Caterpillar and Semilooper Pest Attack     577            577       577      1                1
Healthy_Soyabean                           204            204       204      1                1
Soyabean_Frog_Leaf_Eye                     168            168       168      1                1
Soyabean_Mosaic      

### 1a. Quick check on the single split (optional, ~5 min)
Fine-tune on train, pick on val, look at the test set once.

In [5]:
!python 06_train_classical.py --dataset leaf --img-size 320
!python 07_finetune_cnn.py    --dataset leaf --img-size 320 --model efficientnet_b0 --epochs 30 --patience 8
!python 08_evaluate.py        --dataset leaf --img-size 320 --tta

{'feature_set': 'hand', 'model': 'logreg', 'n_features': 90, 'fit_seconds': 4.9, 'val_acc': 0.6123, 'val_macro_f1': 0.6179}
{'feature_set': 'hand', 'model': 'svc', 'n_features': 90, 'fit_seconds': 2.1, 'val_acc': 0.5543, 'val_macro_f1': 0.5224}
{'feature_set': 'hand', 'model': 'hgb', 'n_features': 90, 'fit_seconds': 1.6, 'val_acc': 0.5145, 'val_macro_f1': 0.4802}
{'feature_set': 'deep', 'model': 'logreg', 'n_features': 960, 'fit_seconds': 1.6, 'val_acc': 0.6159, 'val_macro_f1': 0.6347}
{'feature_set': 'deep', 'model': 'svc', 'n_features': 960, 'fit_seconds': 12.6, 'val_acc': 0.7029, 'val_macro_f1': 0.7024}
{'feature_set': 'deep', 'model': 'hgb', 'n_features': 960, 'fit_seconds': 26.4, 'val_acc': 0.6594, 'val_macro_f1': 0.6512}
{'feature_set': 'both', 'model': 'logreg', 'n_features': 1050, 'fit_seconds': 1.8, 'val_acc': 0.6341, 'val_macro_f1': 0.6369}
{'feature_set': 'both', 'model': 'svc', 'n_features': 1050, 'fit_seconds': 13.9, 'val_acc': 0.721, 'val_macro_f1': 0.7179}
{'feature_set'

### 1b. 5-fold cross-validation (main result, ~25 min)

In [6]:
%%bash
for f in 0 1 2 3 4; do
  echo "=== fold $f ==="
  python 06_train_classical.py --dataset leaf --img-size 320 --fold $f 2>&1 | grep -E "Best on VAL|Traceback|Error"
  python 07_finetune_cnn.py    --dataset leaf --img-size 320 --fold $f --model efficientnet_b0 --epochs 30 --patience 8 2>&1 | grep -E "early stopping|Best val|Traceback|Error"
  python 08_evaluate.py        --dataset leaf --img-size 320 --fold $f --tta 2>&1 | grep -E '"macro_f1"|Traceback|Error'
done

=== fold 0 ===
Best on VAL: deep + svc (macro-F1=0.7095)
early stopping (no val improvement for 8 epochs)
Best val macro-F1 = 0.7363 -> artifacts/leaf/models/fold0/cnn_efficientnet_b0_320.pt  (test set untouched)
  "macro_f1": 0.7229,
  "macro_f1": 0.6709,
=== fold 1 ===
Best on VAL: both + svc (macro-F1=0.6913)
early stopping (no val improvement for 8 epochs)
Best val macro-F1 = 0.7674 -> artifacts/leaf/models/fold1/cnn_efficientnet_b0_320.pt  (test set untouched)
  "macro_f1": 0.7407,
  "macro_f1": 0.6983,
=== fold 2 ===
Best on VAL: both + svc (macro-F1=0.7084)
early stopping (no val improvement for 8 epochs)
Best val macro-F1 = 0.7784 -> artifacts/leaf/models/fold2/cnn_efficientnet_b0_320.pt  (test set untouched)
  "macro_f1": 0.747,
  "macro_f1": 0.7004,
=== fold 3 ===
Best on VAL: deep + svc (macro-F1=0.6448)
Best val macro-F1 = 0.7036 -> artifacts/leaf/models/fold3/cnn_efficientnet_b0_320.pt  (test set untouched)
  "macro_f1": 0.7274,
  "macro_f1": 0.6972,
=== fold 4 ===
Best on

In [7]:
!python 09_cv_summary.py --dataset leaf --img-size 320 --model-tag cnn_efficientnet_b0_320_tta
!python 09_cv_summary.py --dataset leaf --img-size 320 --model-tag classical

fold0: cnn_efficientnet_b0_320_tta_pred.csv
fold1: cnn_efficientnet_b0_320_tta_pred.csv
fold2: cnn_efficientnet_b0_320_tta_pred.csv
fold3: cnn_efficientnet_b0_320_tta_pred.csv
fold4: cnn_efficientnet_b0_320_tta_pred.csv
{
  "model": "cnn_efficientnet_b0_320_tta",
  "folds": 5,
  "n": 2756,
  "per_fold_macro_f1": {
    "fold0": 0.7229,
    "fold1": 0.7407,
    "fold2": 0.747,
    "fold3": 0.7274,
    "fold4": 0.7213
  },
  "mean_fold_macro_f1": 0.7318596272737641,
  "sd_fold_macro_f1": 0.011397037670112221,
  "pooled_macro_f1": 0.7325298694699387,
  "pooled_macro_f1_ci95": [
    0.7164050530636681,
    0.749075124549376
  ],
  "pooled_accuracy": 0.7474600870827286
}

                                        precision    recall  f1-score   support

Caterpillar and Semilooper Pest Attack      0.742     0.693     0.717       577
                      Healthy_Soyabean      0.995     1.000     0.998       204
                Soyabean_Frog_Leaf_Eye      0.523     0.536     0.529       168
    

### 1c. Optional experiments (run only if you want to push the leaf score; compare on the **val** F1 printed by step 7)
```
!python 07_finetune_cnn.py --dataset leaf --img-size 320 --model efficientnet_b0 --epochs 30 --patience 8 --cutmix 1.0
!python 07_finetune_cnn.py --dataset leaf --img-size 320 --model convnext_tiny   --epochs 30 --patience 8 --lr-backbone 1e-4
```
Step 8 picks the best-val checkpoint automatically (`--cnn best`) or averages all of them (`--cnn all`).

In [8]:
!python 07_finetune_cnn.py --dataset leaf --img-size 320 --model efficientnet_b0 --epochs 30 --patience 8 --cutmix 1.0
!python 07_finetune_cnn.py --dataset leaf --img-size 320 --model convnext_tiny   --epochs 30 --patience 8 --lr-backbone 1e-4

epoch 01/30 [head] train_loss=1.5747 val_loss=1.1631 val_acc=0.5145 val_macroF1=0.5332 (15.9s)
epoch 02/30 [head] train_loss=1.3241 val_loss=1.1127 val_acc=0.5435 val_macroF1=0.5504 (3.0s)
epoch 03/30 [full] train_loss=1.4973 val_loss=1.1553 val_acc=0.5399 val_macroF1=0.5468 (44.3s)
epoch 04/30 [full] train_loss=1.2777 val_loss=1.0477 val_acc=0.5978 val_macroF1=0.6168 (8.8s)
epoch 05/30 [full] train_loss=1.1407 val_loss=0.9962 val_acc=0.6196 val_macroF1=0.6344 (8.8s)
epoch 06/30 [full] train_loss=1.1416 val_loss=0.9648 val_acc=0.6848 val_macroF1=0.6870 (8.9s)
epoch 07/30 [full] train_loss=1.0959 val_loss=0.8638 val_acc=0.6884 val_macroF1=0.6863 (9.0s)
epoch 08/30 [full] train_loss=1.0684 val_loss=0.9173 val_acc=0.7210 val_macroF1=0.7123 (9.1s)
epoch 09/30 [full] train_loss=1.0754 val_loss=0.9580 val_acc=0.6920 val_macroF1=0.6901 (9.1s)
epoch 10/30 [full] train_loss=1.0658 val_loss=1.0534 val_acc=0.6630 val_macroF1=0.6723 (9.2s)
epoch 11/30 [full] train_loss=1.1024 val_loss=0.8666 val_a

## 2. UAV dataset
Frames of one video are near-identical and classes come from different days, so: group-aware folds, audit first, no single-split headline number.

In [9]:
!python 01_build_manifest.py --dataset uav --img-size 224 --folds 3
!python 02_cache_images.py   --dataset uav --img-size 224
!python 03_eda.py            --dataset uav --img-size 224
!python 04_extract_features.py --dataset uav --img-size 224
!python 05_audit_split.py    --dataset uav --img-size 224

Found 2842 images in /content/data/MH-SoyaHealthVision An Indian UAV and Leaf Image Dataset for Integrated Crop Health Assessment/Soyabean_UAV-Based_Image_Dataset
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done 740 tasks      | elapsed:   20.3s
[Parallel(n_jobs=-1)]: Done 2840 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done 2842 out of 2842 | elapsed:  1.3min finished

Classes (VERIFY these are your disease / pest / healthy folders):
                                 images  source_groups  clusters  dates  largest_cluster
label                                                                                   
Healthy_Soyabean                    277             49        41      1               24
Soyabean Semilooper_Pest_Attack     786             13        12      4              290
Soyabean_Mosaic                     768              9         9      4              209
rust                               1000            

In [10]:
%%bash
for f in 0 1 2; do
  echo "=== fold $f ==="
  python 06_train_classical.py --dataset uav --img-size 224 --fold $f 2>&1 | grep -E "Best on VAL|Traceback|Error"
  python 07_finetune_cnn.py    --dataset uav --img-size 224 --fold $f --epochs 20 --warmup-epochs 3 --lr-backbone 1e-4 --patience 6 2>&1 | grep -E "early stopping|Best val|Traceback|Error"
  python 08_evaluate.py        --dataset uav --img-size 224 --fold $f --tta 2>&1 | grep -E '"macro_f1"|Traceback|Error'
done

=== fold 0 ===
Best on VAL: both + hgb (macro-F1=0.6411)
early stopping (no val improvement for 6 epochs)
Best val macro-F1 = 0.5822 -> artifacts/uav/models/fold0/cnn_mobilenet_v3_large_224.pt  (test set untouched)
  "macro_f1": 0.9095,
  "macro_f1": 0.9152,
=== fold 1 ===
Best on VAL: hand + hgb (macro-F1=0.8137)
early stopping (no val improvement for 6 epochs)
Best val macro-F1 = 0.7434 -> artifacts/uav/models/fold1/cnn_mobilenet_v3_large_224.pt  (test set untouched)
  "macro_f1": 0.948,
  "macro_f1": 0.9277,
=== fold 2 ===
Best on VAL: both + hgb (macro-F1=0.9904)
early stopping (no val improvement for 6 epochs)
Best val macro-F1 = 0.9862 -> artifacts/uav/models/fold2/cnn_mobilenet_v3_large_224.pt  (test set untouched)
  "macro_f1": 0.75,
  "macro_f1": 0.8541,


In [11]:
!python 09_cv_summary.py --dataset uav --img-size 224 --model-tag cnn
!python 09_cv_summary.py --dataset uav --img-size 224 --model-tag classical

fold0: cnn_mobilenet_v3_large_224_tta_pred.csv
fold1: cnn_mobilenet_v3_large_224_tta_pred.csv
fold2: cnn_mobilenet_v3_large_224_tta_pred.csv
{
  "model": "cnn",
  "folds": 3,
  "n": 2831,
  "per_fold_macro_f1": {
    "fold0": 0.9095,
    "fold1": 0.948,
    "fold2": 0.75
  },
  "mean_fold_macro_f1": 0.8691550460829558,
  "sd_fold_macro_f1": 0.10493016752676544,
  "pooled_macro_f1": 0.8764575951593877,
  "pooled_macro_f1_ci95": [
    0.7367109642805636,
    0.9592944607301732
  ],
  "pooled_accuracy": 0.8795478629459554
}

                                 precision    recall  f1-score   support

               Healthy_Soyabean      0.739     0.993     0.847       277
Soyabean Semilooper_Pest_Attack      0.991     0.880     0.933       786
                Soyabean_Mosaic      0.775     0.952     0.854       768
                           rust      0.968     0.792     0.871      1000

                       accuracy                          0.880      2831
                      macro avg 

In [12]:
!for f in 0 1 2; do python 05_audit_split.py --dataset uav --img-size 224 --fold $f | grep -E "^split|^\(a\)|^\(b\)|^\(c\)"; done

split = fold0: train 1411, val 326, test 1094
(a) val: 0.0% of images share a source group with train
(a) test: 0.0% of images share a source group with train
(b) val: nearest-train similarity 5/25/50/75/95 = [0.901 0.948 0.964 0.973 0.98 ] | 1-NN macro-F1 0.599   <-- near-copies of train: scores will be inflated
(b) test: nearest-train similarity 5/25/50/75/95 = [0.902 0.948 0.968 0.976 0.984] | 1-NN macro-F1 0.867   <-- near-copies of train: scores will be inflated
(c) mean-RGB-only probe, test macro-F1 0.718 (chance 0.25)
split = fold1: train 1991, val 144, test 696
(a) val: 0.0% of images share a source group with train
(a) test: 0.0% of images share a source group with train
(b) val: nearest-train similarity 5/25/50/75/95 = [0.939 0.967 0.974 0.977 0.984] | 1-NN macro-F1 0.741   <-- near-copies of train: scores will be inflated
(b) test: nearest-train similarity 5/25/50/75/95 = [0.925 0.952 0.964 0.973 0.981] | 1-NN macro-F1 0.921   <-- near-copies of train: scores will be inflate

Makes sure CutMix and ConvNeXt runs can't overwrite your baseline checkpoints. It does nothing if you already have the fix.

In [13]:
p = '07_finetune_cnn.py'; s = open(p).read()
old = 'run = f"{args.model}_{args.img_size}"\n'
if '_cutmix' in s:
    print('already fixed - nothing to do')
elif old in s:
    open(p, 'w').write(s.replace(old, 'run = f"{args.model}_{args.img_size}" + ("_cutmix" if args.cutmix > 0 else "")\n')); print('patched')
else:
    print('could not find the line - re-upload the latest soya_pipeline.zip')

patched


Reads existing 320 px cache and resizes it on the GPU, so no re-caching is needed.

In [14]:
import torch, numpy as np, torch.nn.functional as F
NAME, SIZE = 'dinov2_vits14', 224      # later try 'dinov2_vitb14' and/or SIZE = 308 (must be divisible by 14)
FEAT_FILE = f'artifacts/leaf/features/dino_{NAME}_{SIZE}.npy'
OOF_FILE  = f'artifacts/leaf/reports/oof_{NAME}_{SIZE}.csv'

X = np.load('artifacts/leaf/cache/all_x320.npy', mmap_mode='r')
dev = 'cuda'
net = torch.hub.load('facebookresearch/dinov2', NAME).to(dev).eval()
mean = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
out = []
with torch.no_grad(), torch.autocast('cuda'):
    for i in range(0, len(X), 64):
        xb = torch.from_numpy(np.ascontiguousarray(X[i:i + 64])).to(dev).permute(0, 3, 1, 2).float() / 255
        xb = F.interpolate(xb, size=(SIZE, SIZE), mode='bilinear', antialias=True, align_corners=False)
        out.append(net((xb - mean) / std).float().cpu())
E = torch.cat(out).numpy(); np.save(FEAT_FILE, E); print(E.shape)

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 279MB/s]
/tmp/ipykernel_684/3589043310.py:14: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  xb = torch.from_numpy(np.ascontiguousarray(X[i:i + 64])).to(dev).permute(0, 3, 1, 2).float() / 255


(2756, 384)


Uses existing folds. The SVC's C is chosen by a cross-validation inside each training set, so the test fold is never used to pick it.

In [15]:
import numpy as np, pandas as pd
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.metrics import f1_score
from metrics import cluster_bootstrap_f1

E = np.load(FEAT_FILE)
m = pd.read_csv('artifacts/leaf/manifest.csv')
classes = sorted(m.label.unique()); y = m.label_id.to_numpy(); K = len(classes)
P = np.zeros((len(m), K)); fold_f1 = []
for f in range(m.fold.max() + 1):
    te, tr = (m.fold == f).to_numpy(), (m.fold != f).to_numpy()
    gs = GridSearchCV(make_pipeline(StandardScaler(), SVC(class_weight='balanced')), {'svc__C': [1, 10, 100]},
                      cv=StratifiedGroupKFold(3, shuffle=True, random_state=0), scoring='f1_macro', n_jobs=-1)
    gs.fit(E[tr], y[tr], groups=m.cluster[tr])
    C = gs.best_params_['svc__C']
    clf = make_pipeline(StandardScaler(), SVC(C=C, class_weight='balanced', probability=True, random_state=0)).fit(E[tr], y[tr])
    P[te] = clf.predict_proba(E[te])
    fold_f1.append(f1_score(y[te], P[te].argmax(1), average='macro'))
    print(f'fold {f}: C={C}  test macro-F1 {fold_f1[-1]:.4f}')
pred = P.argmax(1)
lo, hi = cluster_bootstrap_f1(y, pred, m.cluster.to_numpy(), 1000, 0)
print(f'\nPOOLED macro-F1 {f1_score(y, pred, average="macro"):.4f}  (95% CI {lo:.3f}-{hi:.3f})  | folds mean {np.mean(fold_f1):.4f}')
print('CNN baseline for comparison: 0.7325 (CI 0.716-0.749)')
pd.DataFrame(P, columns=[f'p_{c}' for c in classes]).assign(relpath=m.relpath, fold=m.fold, label_id=y).to_csv(OOF_FILE, index=False)

fold 0: C=1  test macro-F1 0.6556
fold 1: C=1  test macro-F1 0.6981
fold 2: C=1  test macro-F1 0.6982
fold 3: C=1  test macro-F1 0.6856
fold 4: C=1  test macro-F1 0.6806

POOLED macro-F1 0.6837  (95% CI 0.665-0.702)  | folds mean 0.6836
CNN baseline for comparison: 0.7325 (CI 0.716-0.749)


Averages the two models' probabilities with fixed 50/50 weights. I deliberately don't tune the weights, because that would use the test folds to choose them.

In [16]:
CNN_TAG = 'cnn_efficientnet_b0_320_tta'      # later: 'cnn_ensemble2_tta' or 'cnn_ensemble3_tta'
import numpy as np, pandas as pd
from sklearn.metrics import f1_score
from metrics import cluster_bootstrap_f1

m = pd.read_csv('artifacts/leaf/manifest.csv'); oof = pd.read_csv(OOF_FILE).set_index('relpath')
pc = [c for c in oof.columns if c.startswith('p_')]
parts = []
for f in range(m.fold.max() + 1):
    c = pd.read_csv(f'artifacts/leaf/reports/fold{f}/{CNN_TAG}_pred.csv')
    d = oof.loc[c.relpath]
    parts.append(pd.DataFrame({'y': c.label_id.values, 'cluster': c.cluster.values,
                               'cnn': list(c[pc].to_numpy()), 'dino': list(d[pc].to_numpy())}).assign(fold=f))
d = pd.concat(parts, ignore_index=True)
y, cl = d.y.to_numpy(), d.cluster.to_numpy()
A, B = np.stack(d.cnn), np.stack(d.dino)
for name, p in [('CNN only', A), ('DINOv2 only', B), ('50/50 blend', 0.5 * A + 0.5 * B)]:
    pr = p.argmax(1); lo, hi = cluster_bootstrap_f1(y, pr, cl, 1000, 0)
    print(f'{name:12s} pooled macro-F1 {f1_score(y, pr, average="macro"):.4f}  (95% CI {lo:.3f}-{hi:.3f})')

CNN only     pooled macro-F1 0.7325  (95% CI 0.715-0.750)
DINOv2 only  pooled macro-F1 0.6837  (95% CI 0.665-0.702)
50/50 blend  pooled macro-F1 0.7488  (95% CI 0.732-0.766)


CutMix on every fold, then ensemble

In [17]:
%%bash
for f in 0 1 2 3 4; do
  echo "=== fold $f ==="
  python 07_finetune_cnn.py --dataset leaf --img-size 320 --fold $f --model efficientnet_b0 --epochs 30 --patience 8 --cutmix 1.0 2>&1 | grep -E "Best val|Traceback|Error"
  python 08_evaluate.py --dataset leaf --img-size 320 --fold $f --which cnn --cnn all --tta 2>&1 | grep -E '"model"|"macro_f1"|Traceback|Error'
done
python 09_cv_summary.py --dataset leaf --img-size 320 --model-tag cnn_ensemble2

=== fold 0 ===
Best val macro-F1 = 0.7389 -> artifacts/leaf/models/fold0/cnn_efficientnet_b0_320_cutmix.pt  (test set untouched)
  "model": "cnn_ensemble2_tta",
  "macro_f1": 0.7296,
=== fold 1 ===
Best val macro-F1 = 0.7706 -> artifacts/leaf/models/fold1/cnn_efficientnet_b0_320_cutmix.pt  (test set untouched)
  "model": "cnn_ensemble2_tta",
  "macro_f1": 0.7413,
=== fold 2 ===
Best val macro-F1 = 0.7301 -> artifacts/leaf/models/fold2/cnn_efficientnet_b0_320_cutmix.pt  (test set untouched)
  "model": "cnn_ensemble2_tta",
  "macro_f1": 0.742,
=== fold 3 ===
Best val macro-F1 = 0.7302 -> artifacts/leaf/models/fold3/cnn_efficientnet_b0_320_cutmix.pt  (test set untouched)
  "model": "cnn_ensemble2_tta",
  "macro_f1": 0.7447,
=== fold 4 ===
Best val macro-F1 = 0.7404 -> artifacts/leaf/models/fold4/cnn_efficientnet_b0_320_cutmix.pt  (test set untouched)
  "model": "cnn_ensemble2_tta",
  "macro_f1": 0.7502,
fold0: cnn_ensemble2_tta_pred.csv
fold1: cnn_ensemble2_tta_pred.csv
fold2: cnn_ensembl

add ConvNeXt, then ensemble of three

In [18]:
%%bash
for f in 0 1 2 3 4; do
  echo "=== fold $f ==="
  python 07_finetune_cnn.py --dataset leaf --img-size 320 --fold $f --model convnext_tiny --epochs 30 --patience 8 --lr-backbone 1e-4 2>&1 | grep -E "Best val|Traceback|Error"
  python 08_evaluate.py --dataset leaf --img-size 320 --fold $f --which cnn --cnn all --tta 2>&1 | grep -E '"model"|"macro_f1"|Traceback|Error'
done
python 09_cv_summary.py --dataset leaf --img-size 320 --model-tag cnn_ensemble3

=== fold 0 ===
Best val macro-F1 = 0.7507 -> artifacts/leaf/models/fold0/cnn_convnext_tiny_320.pt  (test set untouched)
  "model": "cnn_ensemble3_tta",
  "macro_f1": 0.7647,
=== fold 1 ===
Best val macro-F1 = 0.7682 -> artifacts/leaf/models/fold1/cnn_convnext_tiny_320.pt  (test set untouched)
  "model": "cnn_ensemble3_tta",
  "macro_f1": 0.7659,
=== fold 2 ===
Best val macro-F1 = 0.7698 -> artifacts/leaf/models/fold2/cnn_convnext_tiny_320.pt  (test set untouched)
  "model": "cnn_ensemble3_tta",
  "macro_f1": 0.7552,
=== fold 3 ===
Best val macro-F1 = 0.7730 -> artifacts/leaf/models/fold3/cnn_convnext_tiny_320.pt  (test set untouched)
  "model": "cnn_ensemble3_tta",
  "macro_f1": 0.7653,
=== fold 4 ===
Best val macro-F1 = 0.7539 -> artifacts/leaf/models/fold4/cnn_convnext_tiny_320.pt  (test set untouched)
  "model": "cnn_ensemble3_tta",
  "macro_f1": 0.7476,
fold0: cnn_ensemble3_tta_pred.csv
fold1: cnn_ensemble3_tta_pred.csv
fold2: cnn_ensemble3_tta_pred.csv
fold3: cnn_ensemble3_tta_pre

## 3. Save and read the results
* Leaf headline = pooled out-of-fold macro-F1 + CI from step 9 (all 2,7xx images, no near-duplicates, group-safe).
* UAV = grouped-CV score **with the step-5 audit next to it**: (a) share of eval frames whose video is in train (should be 0 %), (b) nearest-train similarity / 1-NN F1, (c) mean-RGB probe, (d) leave-one-date-out recall. High (b) or low (d) means the UAV score is in-session only.

In [19]:
!mkdir -p /content/drive/MyDrive/soya_artifacts_v2 && rsync -a --exclude cache artifacts/ /content/drive/MyDrive/soya_artifacts_v2/